In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from config import set_environment
# for the keys - as explained early in chapter 2
set_environment()

# Image generation and understanding

## Image generation

In [2]:
from langchain_community.utilities.dalle_image_generator import DallEAPIWrapper

dalle = DallEAPIWrapper(
    model="dall-e-3", 
    size="1024x1024",
    quality="standard", # [standard, hd]
    n=1, # number of images to generate
)

image_url = dalle.run("A detailed technical diagram of a quantum computer")
print(image_url)

https://oaidalleapiprodscus.blob.core.windows.net/private/org-yVUQit4nbhgTkYZadreOiKAV/user-XL7EZLyfQjVTpLWLXY25Vur4/img-87MMrGOL4vvfWvwFk69gHLyJ.png?st=2025-09-06T14%3A26%3A18Z&se=2025-09-06T16%3A26%3A18Z&sp=r&sv=2024-08-04&sr=b&rscd=inline&rsct=image/png&skoid=b1a0ae1f-618f-4548-84fd-8b16cacd5485&sktid=a48cca56-e6da-484e-a814-9c849652bcb3&skt=2025-09-05T16%3A45%3A15Z&ske=2025-09-06T16%3A45%3A15Z&sks=b&skv=2024-08-04&sig=R6fYr4x3mn/sRipQ1kmeSaCiFJl/QiDbqyY8Ylpq4D0%3D


In [3]:
image_url = dalle.run("POV of an NBA game")
print(image_url)

https://oaidalleapiprodscus.blob.core.windows.net/private/org-yVUQit4nbhgTkYZadreOiKAV/user-XL7EZLyfQjVTpLWLXY25Vur4/img-dqm2jtoO8eNC9JytV5xLJKWi.png?st=2025-09-06T14%3A26%3A50Z&se=2025-09-06T16%3A26%3A50Z&sp=r&sv=2024-08-04&sr=b&rscd=inline&rsct=image/png&skoid=ed3ea2f9-5e38-44be-9a1b-7c1e65e4d54f&sktid=a48cca56-e6da-484e-a814-9c849652bcb3&skt=2025-09-06T15%3A26%3A50Z&ske=2025-09-07T15%3A26%3A50Z&sks=b&skv=2024-08-04&sig=40aS9/UQcB500O9NgWbqTFigGFpfSnnTq4pV9W8hhwQ%3D


In [3]:
from langchain_community.llms import Replicate

# Initialise the text-to-image model:
text2image = Replicate(
    model="stability-ai/stable-diffusion-3.5-large",
    model_kwargs={
        "prompt_strength": 0.8,
        "cfg": 4.5,
        "steps": 40,
        "aspect_ratio": "1:1",
        "output_format": "webp",
        "output_quality": 90,
    }
)

# Generate an image
image_url = text2image.invoke("A detailed technical diagram of an AI agent")

print(image_url)

https://replicate.delivery/xezq/57ZC2vAe5s2nBKQloexe32pRl0gsLXN1Anzc5U6B9bYk7AlqA/out-0.webp


In [7]:
# Generate an image
image_url = text2image.invoke("fat tabby stealing jewelleries from a skinny white cat")

print(image_url)

https://replicate.delivery/xezq/NcbHN2pN5KY2J5khTfUNDR2GW4JsjJCMOYziy2P4nFrsPQpKA/out-0.webp


## Image understanding

In [8]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

def analyze_image(image_url: str, question: str) -> str:
    chat = ChatOpenAI(model="gpt-4o-mini", max_tokens=256)

    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": question
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": image_url,
                    "detail": "auto"
                }
            }
        ]
    )

    response = chat.invoke([message])
    return response.content

# Example usage
image_url = "https://static01.nyt.com/athletic/uploads/wp/2022/04/27020535/GettyImages-1240264153-scaled-e1651074796467.jpg"
questions = [
    "What objects do you see in this image?",
    "What is the overall mood or atmosphere?",
    "Are there any people in the image?"
]

for question in questions:
    print(f"\nQ: {question}")
    print(f"A: {analyze_image(image_url, question)}")


Q: What objects do you see in this image?
A: In the image, I see several basketball players on a court. One player appears to be making a jump shot while holding a basketball. The players are in jerseys, with one team wearing dark blue and the other wearing white. The court features markings and a logo, likely of a basketball team. There's also a basketball hoop visible, with the net in the foreground. The stands are filled with spectators in the background.

Q: What is the overall mood or atmosphere?
A: The overall mood of the image exudes excitement and intensity, typical of a high-stakes basketball game. The dynamic action of a player dunking the ball suggests a moment of triumph and athletic prowess, likely eliciting cheers from the crowd. The atmosphere is electric, reflecting the competitive spirit and energy that characterizes live sports events.

Q: Are there any people in the image?
A: Yes, there are people in the image. It appears to depict a basketball scene, likely featuri

### With Gemini

In [10]:
import base64
from langchain_google_genai import ChatGoogleGenerativeAI

with open("stable-diffusion.png", 'rb') as image_file:
    image_bytes = image_file.read()
    base64_bytes = base64.b64encode(image_bytes).decode('utf-8')

prompt = [
    {"type": "text", "text": "Describe the image:"},
    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_bytes}"}}
]

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

response = llm.invoke([HumanMessage(content=prompt)])
print(response.content)


The image presents a striking, futuristic half-body portrait of an android or robot, set against a deep blue, high-tech digital background.

The robot is positioned slightly off-center to the right, facing left in a profile view, with its gaze directed forward. Its head is encased in a sleek, segmented white shell with a glossy finish, revealing a smooth, dark, almost black humanoid face. Piercing, luminous orange eyes glow intensely, drawing the viewer's attention. On the right side of its head, a prominent circular mechanism glows with a vibrant blue light, suggesting an advanced sensor or processing unit. Intricate wiring and internal components, some glowing with small yellow lights, are visible around its neck and within the gaps of its white plating, particularly around the shoulders and chest, hinting at complex internal machinery. The body continues the design with segmented white armor-like plating, contrasting with the dark, complex machinery and glowing internal elements ben

In [11]:
import base64
with open("stable-diffusion.png", 'rb') as image_file:
 image_bytes = image_file.read()
 base64_bytes = base64.b64encode(image_bytes).decode("utf-8")


prompt = [
   {"type": "text", "text": "Describe the image: "},
   {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_bytes}"}},
]


llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-pro",
    temperature=0,
)
response = llm.invoke([HumanMessage(content=prompt)])
print(response.content)

The image presents a futuristic, female-appearing android or cyborg against a backdrop of glowing blue digital displays. The android's head and upper torso are visible. Its design is a blend of smooth, white plating and exposed, intricate dark metallic components, suggesting advanced robotics. Small, bright yellow/orange lights dot the internal mechanisms, giving a sense of active energy flow within the machine.

The android's head is rounded, with a dark, visor-like face featuring glowing orange eyes.  A large, circular feature on the side of its head resembles a complex sensor or audio device, also with a glowing blue center. The neck and shoulder area reveal a complex network of cables, joints, and mechanical parts, suggesting flexibility and a wide range of motion.

The background displays appear to be holographic or projected interfaces, each containing various graphs, charts, and data visualizations. The dominant color is bright blue, with data represented in lighter blue, white,

#### Using Google Cloud Storage

In [ ]:
prompt = [
   {"type": "text", "text": "Describe the video in a few sentences."},
   {"type": "media", "file_uri": video_uri, "mime_type": "video/mp4"},
]


response = llm.invoke([HumanMessage(content=prompt)])
print(response.content)